# 9. Conclusiones y limitaciones

In [76]:
mejor_ext = m_clf.loc[m_clf["auc_pr_media"].idxmax()]
display(Markdown(f"""## 9. Conclusiones y limitaciones

### 9.1 Conclusiones

1. **Modelos y balanceo.** La mejor combinación en la validación anidada es {etiqueta(mejor_ext)}, con un AUC-PR externo de {num(mejor_ext['auc_pr_media'], 3)} ± {num(mejor_ext['auc_pr_de'], 3)}; dentro de 2008 ningún modelo discrimina y la señal aparece al validar en 2009, donde los modelos simples extrapolan igual o mejor que los flexibles. En prueba, el mayor AUC-PR entre los finalistas es el de {mejor_te['Modelo']} ({num(mejor_te['AUC-PR'], 3)} {mejor_te['AUC-PR IC 95 %']}), {'por debajo del' if refe['AUC-PR'] > mejor_te['AUC-PR'] else 'frente al'} {num(refe['AUC-PR'], 3)} del modelo base del Entregable 2 (logística L2 con C = 0,01). {'Es decir, las 165 combinaciones no superan a una logística bien regularizada: en un problema con cambio de régimen, la regularización y la simplicidad pesan más que la flexibilidad y la búsqueda de hiperparámetros. ' if refe['AUC-PR'] >= mejor_te['AUC-PR'] else ''}Las diferencias entre los mejores finalistas son del orden de la variabilidad entre periodos: el diagrama de diferencia crítica no separa a ninguno con significancia {'y DeLong no confirma ninguna de las ' + str(len(dl_t)) + ' diferencias dirigidas' if (dl_t['p Holm'] >= 0.05).all() else 'y DeLong confirma ' + str(int((dl_t['p Holm'] < 0.05).sum())) + ' de ' + str(len(dl_t)) + ' diferencias dirigidas'} tras el ajuste de Holm.
2. **Balanceo.** SMOTE, ADASYN y `class_weight` elevan el recall y el F1 con el umbral por defecto, pero no mejoran el orden de los clientes (AUC-PR) y empeoran la calibración: actúan como un desplazamiento del umbral. El cambio de prevalencia entre el entrenamiento ({pct(y_tr.mean())}) y la prueba ({pct(y_te.mean())}) es un problema distinto, que solo la recalibración con los meses cerrados corrige.
3. **Optimización.** Con 20 configuraciones por pliegue, Friedman sobre la métrica externa da {p_txt(fr_clf['p'])} entre los cuatro métodos de la guía, y sobre el área bajo la curva *anytime* {p_txt(fr_auac['p'])}. El área bajo la curva *anytime* es de {num(auac_m['grid'], 3)} con Grid Search, {num(auac_m['random'], 3)} con Random Search, {num(auac_m['bayes'], 3)} con la bayesiana y {num(auac_m['genetico'], 3)} con el genético: los métodos adaptativos llegan antes a buenas soluciones que la rejilla (delta de Cliff de {num(b_vs['delta de Cliff'].iloc[0], 2)} y {num(g_vs['delta de Cliff'].iloc[0], 2)}), pero frente a Random Search la ventaja es pequeña ({num(b_vs['delta de Cliff'].iloc[1], 2)} y {num(g_vs['delta de Cliff'].iloc[1], 2)}), porque la fase aleatoria inicial consume buena parte de un presupuesto de 20 configuraciones. La mejor búsqueda tampoco se traduce en mejores modelos en el bucle externo de forma apreciable. GOAT iterativo, como extensión, usa una mediana de {np.median([c['n_eval'] for c in goat_c]):.0f} evaluaciones sin diferencia significativa frente a Optuna ({p_txt(h1_clf['p Wilcoxon'])}), lo que sostiene H1 en cuanto al ahorro de evaluaciones; H2 (que GOAT pierda en la SVM) no se confirma ({p_txt(h2['p Wilcoxon'])}). Successive Halving examina {int(hv['n_eval'].mean())} configuraciones por pliegue con un costo equivalente a {num(hv['eval_equivalentes'].mean(), 1)} evaluaciones completas.
4. **Costo computacional.** El cuello de botella real es el ajuste de los modelos ({num(pc.set_index('Componente').loc['Ajuste de XGBoost (booster)', '% del total'], 0)} % del tiempo en el perfil de XGBoost), seguido del preprocesamiento ({num(pc.set_index('Componente').loc['Preprocesamiento (ColumnTransformer)', '% del total'], 0)} %); el costo propio del optimizador es despreciable. Los histogramas de XGBoost (unas {num(ex['Ajuste (s)'] / hi_['Ajuste (s)'], 0)} veces más rápidos sin perder desempeño), la aproximación de Nyström en la SVM y la paralelización al nivel de tarea son las optimizaciones con mayor efecto; SAGA y los índices espaciales de KNN no compensan con 49 columnas, y FAISS solo conviene cuando la inferencia es masiva.
5. **Regresión.** Con información previa a la llamada, la duración apenas se puede pronosticar: el mejor R² en prueba es {num(mejor_r['R²'], 3)}. Los modelos superan a la media histórica de forma significativa (Clark-West {p_txt(cw['p'])}, SPA {p_txt(spa_p['consistent'])}) y el Model Confidence Set conserva {len(en_mcs)} de {len(mcs)} modelos{', sin la media' if 'Media histórica (referencia)' not in en_mcs.index else ''}, pero la ganancia es demasiado pequeña para planificar llamada a llamada; es coherente con la sección 3.5: la duración depende sobre todo de lo que ocurre durante la llamada.

### 9.2 Limitaciones

- **Cambio de régimen.** Los pliegues de 2008 casi no permiten discriminar y la prueba (2009-2010) tiene otra prevalencia; las métricas del bucle externo y las de prueba no son comparables en nivel, y la media ± desviación estándar mezcla variabilidad de muestreo y cambio de régimen.
- **Presupuesto corto.** Veinte configuraciones por pliegue limitan la ventaja de los métodos adaptativos y la exploración del genético, cuya población converge en pocas generaciones; las conclusiones sobre eficiencia valen para este presupuesto y espacios de dos a cuatro dimensiones.
- **Pocos bloques.** Cinco pliegues externos dan poca potencia a Friedman y una diferencia crítica de Nemenyi amplia; los bloques mensuales complementan, pero no son independientes.
- **Sobremuestreo en el espacio *one-hot*.** SMOTE y ADASYN interpolan columnas binarias; SMOTE-NC sería más apropiado para variables categóricas.
- **GOAT iterativo** es una extensión propia sin réplicas independientes; el método publicado es de un solo pase.
- **Entorno.** No se usó GPU: la máquina no tiene GPU NVIDIA y, con unos 41 000 registros, no se justifica, porque XGBoost con histogramas ajusta en {num(hi_['Ajuste (s)'], 2)} s en CPU y scikit-learn está diseñado para CPU. Tampoco se usaron Dask ni Ray, porque los datos caben en memoria; los tiempos son de una sola máquina de 12 núcleos.
- **Objetivo de regresión.** `log1p(duration)` se eligió por ser la única variable continua con sentido a nivel de cliente; su falta de señal limita el valor de la rama de regresión como comparación entre modelos.
"""))

## 9.1 Conclusiones

1. **Modelos y balanceo.** La mejor combinación en la validación anidada es Naive Bayes · class_weight · Bayesiana (Optuna TPE), con un AUC-PR externo de 0,103 ± 0,108; dentro de 2008 ningún modelo discrimina y la señal aparece al validar en 2009, donde los modelos simples extrapolan igual o mejor que los flexibles. En prueba, el mayor AUC-PR entre los finalistas es el de SVM (0,535 [0,513; 0,554]), por debajo del 0,548 del modelo base del Entregable 2 (logística L2 con C = 0,01). Es decir, las 165 combinaciones no superan a una logística bien regularizada: en un problema con cambio de régimen, la regularización y la simplicidad pesan más que la flexibilidad y la búsqueda de hiperparámetros. Las diferencias entre los mejores finalistas son del orden de la variabilidad entre periodos: el diagrama de diferencia crítica no separa a ninguno con significancia y DeLong no confirma ninguna de las 4 diferencias dirigidas tras el ajuste de Holm.
2. **Balanceo.** SMOTE, ADASYN y `class_weight` elevan el recall y el F1 con el umbral por defecto, pero no mejoran el orden de los clientes (AUC-PR) y empeoran la calibración: actúan como un desplazamiento del umbral. El cambio de prevalencia entre el entrenamiento (6,4 %) y la prueba (30,8 %) es un problema distinto, que solo la recalibración con los meses cerrados corrige.
3. **Optimización.** Con 20 configuraciones por pliegue, Friedman sobre la métrica externa da p = 0,025 entre los cuatro métodos de la guía, y sobre el área bajo la curva *anytime* p < 0,001. El área bajo la curva *anytime* es de 0,661 con Grid Search, 0,745 con Random Search, 0,763 con la bayesiana y 0,772 con el genético: los métodos adaptativos llegan antes a buenas soluciones que la rejilla (delta de Cliff de 0,32 y 0,34), pero frente a Random Search la ventaja es pequeña (0,04 y 0,11), porque la fase aleatoria inicial consume buena parte de un presupuesto de 20 configuraciones. La mejor búsqueda tampoco se traduce en mejores modelos en el bucle externo de forma apreciable. GOAT iterativo, como extensión, usa una mediana de 16 evaluaciones sin diferencia significativa frente a Optuna (p = 0,267), lo que sostiene H1 en cuanto al ahorro de evaluaciones; H2 (que GOAT pierda en la SVM) no se confirma (p = 0,729). Successive Halving examina 39 configuraciones por pliegue con un costo equivalente a 8,9 evaluaciones completas.
4. **Costo computacional.** El cuello de botella real es el ajuste de los modelos (69 % del tiempo en el perfil de XGBoost), seguido del preprocesamiento (19 %); el costo propio del optimizador es despreciable. Los histogramas de XGBoost (unas 10 veces más rápidos sin perder desempeño), la aproximación de Nyström en la SVM y la paralelización al nivel de tarea son las optimizaciones con mayor efecto; SAGA y los índices espaciales de KNN no compensan con 49 columnas, y FAISS solo conviene cuando la inferencia es masiva.
5. **Regresión.** Con información previa a la llamada, la duración apenas se puede pronosticar: el mejor R² en prueba es 0,088. Los modelos superan a la media histórica de forma significativa (Clark-West p < 0,001, SPA p = 0,003) y el Model Confidence Set conserva 9 de 34 modelos, sin la media, pero la ganancia es demasiado pequeña para planificar llamada a llamada; es coherente con la sección 3.5: la duración depende sobre todo de lo que ocurre durante la llamada.

## 9.2 Limitaciones

- **Cambio de régimen.** Los pliegues de 2008 casi no permiten discriminar y la prueba (2009-2010) tiene otra prevalencia; las métricas del bucle externo y las de prueba no son comparables en nivel, y la media ± desviación estándar mezcla variabilidad de muestreo y cambio de régimen.
- **Presupuesto corto.** Veinte configuraciones por pliegue limitan la ventaja de los métodos adaptativos y la exploración del genético, cuya población converge en pocas generaciones; las conclusiones sobre eficiencia valen para este presupuesto y espacios de dos a cuatro dimensiones.
- **Pocos bloques.** Cinco pliegues externos dan poca potencia a Friedman y una diferencia crítica de Nemenyi amplia; los bloques mensuales complementan, pero no son independientes.
- **Sobremuestreo en el espacio *one-hot*.** SMOTE y ADASYN interpolan columnas binarias; SMOTE-NC sería más apropiado para variables categóricas.
- **GOAT iterativo** es una extensión propia sin réplicas independientes; el método publicado es de un solo pase.
- **Entorno.** No se usó GPU: la máquina no tiene GPU NVIDIA y, con unos 41 000 registros, no se justifica, porque XGBoost con histogramas ajusta en 1,87 s en CPU y scikit-learn está diseñado para CPU. Tampoco se usaron Dask ni Ray, porque los datos caben en memoria; los tiempos son de una sola máquina de 12 núcleos.
- **Objetivo de regresión.** `log1p(duration)` se eligió por ser la única variable continua con sentido a nivel de cliente; su falta de señal limita el valor de la rama de regresión como comparación entre modelos.
